# Model Evaluation with DeepEval — Logged to MLflow

Same workflow as `model_eval_deepeval.ipynb` (golden dataset → generate → DeepEval metrics → results), but the run is logged to an **MLflow** tracking server instead of only being saved as a local JSON artifact — MLflow is this repo's system of record for offline eval runs (see `CLAUDE.md`).

It reuses `genai_eval.mlflow_logging.log_eval_run` — the same logging function the CLI runner (`python -m genai_eval.eval_runner`) uses — so a run started from this notebook and a run started from the CLI land in MLflow with the same shape and are directly comparable.

Every run logs `judge_model` **and** `judge_prompt_ver` as params (the pinned-judge rule in `CLAUDE.md`): a judge change invalidates baselines, so the exact judge configuration must travel with every score.

## Table of contents

1. [Requirements](#1-requirements)
2. [Configuration](#2-configuration)
   - 2a. [Model under test](#2a-model-under-test)
   - 2b. [Judge configuration](#2b-judge-configuration)
   - 2c. [MLflow configuration](#2c-mlflow-configuration)
3. [Golden dataset](#3-golden-dataset)
4. [Generate outputs from the model under test](#4-generate-outputs-from-the-model-under-test)
5. [Build test cases & metrics](#5-build-test-cases--metrics)
6. [Run the evaluation](#6-run-the-evaluation)
7. [Results](#7-results)
8. [Log to MLflow](#8-log-to-mlflow)

## 1. Requirements

- `pip install -r notebooks/requirements.txt` (includes `deepeval`, `openai`, `mlflow`).
- A reachable **MLflow tracking server** (`MLFLOW_TRACKING_URI`) — local (`mlflow server`) or self-hosted, per this repo's air-gap-friendly setup. See `docs/` for standing one up.

In [ ]:
%pip install -q -r requirements.txt

## 2. Configuration

Three independent things to configure: the **model under test** (2a), the **judge model** (2b), and the **MLflow tracking server** to log the run to (2c).

### 2a. Model under test

A reachable **OpenAI-compatible endpoint** serving the model under test — same assumption as every scenario in this repo.

In [ ]:
import json
import os
from pathlib import Path

# Notebook is expected to run from notebooks/ inside the repo (Jupyter's default cwd).
REPO_ROOT = Path.cwd().parent if not (Path.cwd() / "notebooks" / "sample-prompts").exists() else Path.cwd()
assert (REPO_ROOT / "notebooks" / "sample-prompts").exists(), (
    f"Could not find the repo root from {Path.cwd()} — run this notebook from the notebooks/ directory."
)

# ---- Model under test (any OpenAI-compatible endpoint) -------------------
MODEL = ""        # model name the endpoint serves, e.g. "Qwen/Qwen3-0.6B"
BASE_URL = ""     # e.g. "http://localhost:8000/v1"  (note the /v1)
UNDER_TEST_API_KEY = os.environ.get("UNDER_TEST_API_KEY", "none")  # most local endpoints ignore it
TEMPERATURE = 0.0  # deterministic-ish outputs -> more reproducible scores
MAX_TOKENS = 300

assert MODEL and BASE_URL, "Set MODEL and BASE_URL above."
print(f"Model under test: {MODEL} @ {BASE_URL}")

### 2b. Judge configuration

A **judge model** for the LLM-as-judge metrics. Pick one of two backends by setting `JUDGE_BACKEND` below — no out-of-band CLI state, everything is configured in this cell and recorded in the MLflow run:

- **`"openai"`** — a hosted OpenAI judge. Needs `OPENAI_API_KEY` in the environment; `JUDGE_MODEL` names the judge (e.g. `gpt-4o-mini`).
- **`"local"`** — any OpenAI-compatible endpoint (local NIM/vLLM/TGI, or another cloud). Set `JUDGE_MODEL`, `JUDGE_BASE_URL`, and (if needed) `JUDGE_API_KEY`. Nothing leaves your network.

The next cell turns whichever backend you pick into a single judge object that all the metrics share, so switching judges is a one-line change here.

> **Judge and model-under-test are fully independent** — different backends, endpoints, and models. Use a judge that is **stronger than (and different from) the model under test** — a model grading its own homework inflates scores.

> **Pinned-judge rule** (`CLAUDE.md`): `judge_model` and `judge_prompt_ver` are logged with every MLflow run (Section 8). A judge change invalidates baselines — it is not a drop-in swap for comparing runs.

> **Local-judge JSON errors.** DeepEval asks the judge for strict JSON and parses it; weak judges wrap it in prose or ```` ```json ```` fences, giving `ValueError: Evaluation LLM outputted an invalid JSON`. The local judge below defends against this by requesting **structured output** and returning a validated object, with a lenient JSON-recovery fallback. If you still hit it, the real fix is a **more capable judge** — a 7B+ instruct model handles the JSON contract reliably; sub-1B models often don't. Section 6's per-case `try`/`except` around `measure()` keeps one bad case from aborting the whole run.

In [ ]:
# ---- Judge model (used by the DeepEval metrics) --------------------------
# Pick the backend, then fill in the fields it uses. The next cell builds a
# single judge object from these — nothing depends on `deepeval` CLI state.
JUDGE_BACKEND = "openai"           # "openai" (hosted) or "local" (any OpenAI-compatible endpoint)

JUDGE_MODEL = "gpt-4o-mini"        # judge model name (both backends)

# Only used when JUDGE_BACKEND == "local":
JUDGE_BASE_URL = "http://localhost:8001/v1"                   # judge endpoint (note the /v1)

# Some judge endpoints (hosted OpenAI, or a local endpoint with auth enabled)
# require an API key. Set it here — if provided, it's exported to the
# environment so JUDGE_API_KEY below picks it up (and OPENAI_API_KEY, for the
# JUDGE_BACKEND == "openai" path).
API_KEY = 'API_KEY_SET_HERE'

if API_KEY:
    os.environ["JUDGE_API_KEY"] = API_KEY
    os.environ["OPENAI_API_KEY"] = API_KEY

JUDGE_API_KEY = os.environ.get("JUDGE_API_KEY", "none")       # most local endpoints ignore it

assert JUDGE_BACKEND in ("openai", "local"), "JUDGE_BACKEND must be 'openai' or 'local'."
if JUDGE_BACKEND == "openai" and not os.environ.get("OPENAI_API_KEY"):
    print("WARNING: JUDGE_BACKEND='openai' but OPENAI_API_KEY is not set — "
          "either export it, or switch JUDGE_BACKEND to 'local'.")

if JUDGE_BACKEND == "openai":
    print(f"Judge: {JUDGE_MODEL} (OpenAI)")
else:
    print(f"Judge: {JUDGE_MODEL} @ {JUDGE_BASE_URL} (local / OpenAI-compatible)")

In [ ]:
# Build ONE judge object from the config above; every metric shares it.
#
# - "openai": pass the model-name string straight through — deepeval's default
#   path, uses OPENAI_API_KEY.
# - "local":  wrap any OpenAI-compatible endpoint in a tiny DeepEvalBaseLLM.
#   We talk to it with the same `openai` client the rest of the repo uses, so
#   there's no dependency on `deepeval set-local-model` / CLI global state.
#
# Why the `schema` handling below matters: deepeval metrics ask the judge for
# strict JSON, then parse the raw string. Smaller/local judges often wrap it in
# prose or ```json fences, which makes deepeval raise
#   "Evaluation LLM outputted an invalid JSON. Please use a better model."
# deepeval avoids that entirely if generate(prompt, schema=Model) returns a
# Pydantic instance — it then trusts the object instead of parsing text. We do
# that via the OpenAI structured-output path, with a lenient fallback for
# endpoints that don't support it.

if JUDGE_BACKEND == "openai":
    judge = JUDGE_MODEL  # metric(model="gpt-4o-mini") — deepeval's built-in OpenAI path
else:
    import json as _json
    import re as _re
    from openai import OpenAI
    from deepeval.models.base_model import DeepEvalBaseLLM

    def _coerce_json(text: str) -> dict:
        """Best-effort recovery of a JSON object from a chatty judge reply."""
        text = text.strip()
        fence = _re.match(r"^```(?:json)?\s*(.*?)\s*```$", text, _re.DOTALL)
        if fence:
            text = fence.group(1).strip()
        try:
            return _json.loads(text)
        except _json.JSONDecodeError:
            start, end = text.find("{"), text.rfind("}")
            if start != -1 and end > start:
                return _json.loads(text[start:end + 1])
            raise

    class OpenAICompatibleJudge(DeepEvalBaseLLM):
        """Judge backed by any OpenAI-compatible /v1/chat/completions endpoint.

        Honors deepeval's optional `schema` (a Pydantic model): when present we
        request structured output and return a model instance, so deepeval never
        has to parse the raw string — this is what fixes the "invalid JSON" error
        for weaker local judges.
        """

        def __init__(self, model: str, base_url: str, api_key: str):
            self.model = model
            self._client = OpenAI(base_url=base_url, api_key=api_key)

        def load_model(self):
            return self._client

        def _complete(self, prompt: str, schema=None):
            messages = [{"role": "user", "content": prompt}]
            if schema is None:
                resp = self._client.chat.completions.create(
                    model=self.model, messages=messages, temperature=0.0,
                )
                return (resp.choices[0].message.content or "").strip()

            try:
                parsed = self._client.beta.chat.completions.parse(
                    model=self.model, messages=messages, temperature=0.0,
                    response_format=schema,
                ).choices[0].message.parsed
                if parsed is not None:
                    return parsed
            except Exception:
                pass

            try:
                resp = self._client.chat.completions.create(
                    model=self.model, messages=messages, temperature=0.0,
                    response_format={"type": "json_object"},
                )
            except Exception:
                messages = [{"role": "user",
                             "content": prompt + "\n\nRespond with ONLY a valid JSON object."}]
                resp = self._client.chat.completions.create(
                    model=self.model, messages=messages, temperature=0.0,
                )
            raw = (resp.choices[0].message.content or "").strip()
            return schema.model_validate(_coerce_json(raw))

        def generate(self, prompt: str, schema=None, *args, **kwargs):
            return self._complete(prompt, schema)

        async def a_generate(self, prompt: str, schema=None, *args, **kwargs):
            return self._complete(prompt, schema)

        def get_model_name(self) -> str:
            return self.model

    judge = OpenAICompatibleJudge(JUDGE_MODEL, JUDGE_BASE_URL, JUDGE_API_KEY)

print(f"Judge object ready: {judge if isinstance(judge, str) else judge.get_model_name()}")

In [ ]:
# Preflight: confirm the judge endpoint is reachable and answers BEFORE we build
# the dataset and spend a full eval on it.
def _check_judge():
    if JUDGE_BACKEND == "openai":
        from openai import OpenAI
        reply = OpenAI().chat.completions.create(
            model=judge,
            messages=[{"role": "user", "content": "Reply with the single word: ok"}],
            max_tokens=5,
        ).choices[0].message.content
        return f"OpenAI judge '{judge}'", reply
    else:
        reply = judge.generate("Reply with the single word: ok")
        return f"local judge '{judge.get_model_name()}' @ {JUDGE_BASE_URL}", reply

try:
    who, reply = _check_judge()
    print(f"OK — {who} is reachable.")
    print(f"     sample reply: {reply!r}")
except Exception as e:
    raise RuntimeError(
        f"Judge endpoint preflight FAILED for JUDGE_BACKEND={JUDGE_BACKEND!r}. "
        "Fix the judge config in Section 2b before continuing — check "
        + ("OPENAI_API_KEY and JUDGE_MODEL." if JUDGE_BACKEND == "openai"
           else f"JUDGE_BASE_URL ({JUDGE_BASE_URL!r}, note the /v1), JUDGE_MODEL, "
                "and that the endpoint is up.")
        + f"\nUnderlying error: {type(e).__name__}: {e}"
    ) from e

### 2c. MLflow configuration

Where this run gets logged. Same env vars as the CLI runner (`MLFLOW_TRACKING_URI`, `MLFLOW_EXPERIMENT`) so a run started from this notebook is indistinguishable in MLflow from one started via `python -m genai_eval.eval_runner`.

In [ ]:
MLFLOW_TRACKING_URI = os.environ.get("MLFLOW_TRACKING_URI", "http://localhost:5000")
MLFLOW_EXPERIMENT = os.environ.get("MLFLOW_EXPERIMENT", "genai-eval")

assert MLFLOW_TRACKING_URI, "Set MLFLOW_TRACKING_URI above (or export it before starting Jupyter)."
print(f"MLflow tracking URI: {MLFLOW_TRACKING_URI}")
print(f"MLflow experiment:   {MLFLOW_EXPERIMENT}")

## 3. Golden dataset

Loaded from this repo's golden dataset, `datasets/golden_qa_de.jsonl` — one JSON object per line, following the schema in `CLAUDE.md`: `id`, `language`, `category`, `prompt`, `max_tokens`, `temperature`, plus optional `expected_output`/`contexts`/`metric_set`.

`expected_output` is a *reference answer*, not a string to match verbatim — the GEval judge compares substance (facts, coverage, intent), not wording.

In [ ]:
DATASET_PATH = REPO_ROOT / "datasets" / "golden_qa_de.jsonl"

eval_dataset = []
with open(DATASET_PATH, encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        record = json.loads(line)
        eval_dataset.append({
            "input": record["prompt"],
            "expected_output": record.get("expected_output"),
        })

print(f"{len(eval_dataset)} golden test cases loaded from {DATASET_PATH}:")
for i, c in enumerate(eval_dataset):
    print(f"  [{i}] {c['input'][:70]}...")

## 4. Generate outputs from the model under test

One chat completion per test case, `temperature=0` for repeatability. This is the only cell that talks to the model under test — everything after it is judging.

In [ ]:
from openai import OpenAI

client = OpenAI(base_url=BASE_URL, api_key=UNDER_TEST_API_KEY)

def generate(prompt: str) -> str:
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=TEMPERATURE,
        max_tokens=MAX_TOKENS,
    )
    return (resp.choices[0].message.content or "").strip()

for i, case in enumerate(eval_dataset):
    case["actual_output"] = generate(case["input"])
    print(f"[{i}] {case['input'][:50]}...")
    print(f"    -> {case['actual_output'][:100]}...\n")

## 5. Build test cases & metrics

Each `LLMTestCase` bundles `input` + `actual_output` (+ `expected_output` for reference-based metrics). Two metrics, both LLM-as-judge, both returning a 0–1 score with a written reason:

| Metric | What it checks | Needs `expected_output` | Threshold |
|---|---|---|---|
| **Correctness** (GEval, custom criteria) | actual output is substantively consistent with the reference answer | yes | 0.5 |
| **Answer Relevancy** (built-in) | output actually addresses the input, no off-topic filler | no | 0.7 |

A case **passes** a metric when its score ≥ the metric's threshold. Both metrics use the shared `judge` object built in Section 2b.

These thresholds and metric names match `docs/metric-registry.md` — the same `correctness`/`answer_relevancy` names owned by DeepEval elsewhere in this repo.

In [ ]:
from deepeval.test_case import LLMTestCase
from deepeval.metrics import GEval, AnswerRelevancyMetric

# The evaluation-params enum was renamed in newer deepeval releases
# (LLMTestCaseParams -> SingleTurnParams); support both.
try:
    from deepeval.test_case import LLMTestCaseParams as Params
except ImportError:
    from deepeval.test_case import SingleTurnParams as Params

test_cases = [
    LLMTestCase(
        input=c["input"],
        actual_output=c["actual_output"],
        expected_output=c["expected_output"],
    )
    for c in eval_dataset
]

# `judge` (built in Section 2b) is either an OpenAI model-name string or a
# DeepEvalBaseLLM instance — metric(model=...) accepts both. Passing the object
# (not JUDGE_MODEL) is what keeps a local judge from falling back to OpenAI.
#
# Factories, not shared instances: DeepEval metrics hold score/reason state
# after measure(), so Section 6 builds one fresh metric per (case, metric)
# from these — same pattern as genai_eval.eval_runner.REGISTRY. Each factory is
# paired with its own display name rather than read off the metric instance —
# built-in metrics like AnswerRelevancyMetric don't expose a `.name` attribute
# (only `GEval` does, since it's passed in explicitly), so `metric.name` raises
# AttributeError for them.
def make_correctness():
    return GEval(
        name="Correctness",
        criteria=(
            "Determine whether the actual output is factually and substantively "
            "consistent with the expected output. Penalize missing key facts, "
            "contradictions, and unmet instructions from the input; do NOT penalize "
            "differences in wording, formatting, or reasonable extra detail."
        ),
        evaluation_params=[Params.INPUT, Params.ACTUAL_OUTPUT, Params.EXPECTED_OUTPUT],
        threshold=0.5,
        model=judge,
    )


def make_relevancy():
    return AnswerRelevancyMetric(threshold=0.7, model=judge)


metric_factories = [("Correctness", make_correctness), ("Answer Relevancy", make_relevancy)]

print(f"{len(test_cases)} test cases, 2 metrics (Correctness, Answer Relevancy)")

## 6. Run the evaluation

Scores every metric against every test case (each score = one or more judge calls, so expect ~a few seconds per case).

In [ ]:
# Score each test case with each metric via metric.measure() directly — the
# same pattern genai_eval.eval_runner.run_evaluation uses — rather than
# deepeval's evaluate() wrapper. Two reasons:
#   1. evaluate()'s error-handling kwarg has drifted across deepeval releases
#      (ErrorConfig(ignore_errors=...), a flat ignore_errors=..., or neither),
#      so a version-agnostic call is a moving target.
#   2. evaluate()'s internal async orchestration can wedge a Jupyter kernel's
#      already-running event loop ("RuntimeError: cannot enter context ...
#      already entered"). measure() runs synchronously per case, avoiding it.
# A fresh metric instance per (case, metric) avoids state bleed between cases;
# a per-case try/except means one bad case (e.g. an unrecoverable non-JSON
# reply from a weak local judge) can't abort the run.
rows = []
for i, tc in enumerate(test_cases):
    for metric_name, factory in metric_factories:
        metric = factory()
        try:
            metric.measure(tc)
            score, reason, passed = metric.score, metric.reason, metric.is_successful()
        except Exception as exc:
            score, reason, passed = None, f"{type(exc).__name__}: {exc}", False
            print(f"[{i}] {metric_name} FAILED: {reason}")
        rows.append({
            "case": i,
            "input": eval_dataset[i]["input"][:60] + "...",
            "metric": metric_name,
            "score": round(score, 3) if score is not None else None,
            "passed": passed,
            "reason": reason,
        })

print(f"Scored {len(test_cases)} cases x {len(metric_factories)} metrics.")

## 7. Results

Flatten into one row per (test case, metric): score, pass/fail, and the judge's reason — the reason is the debugging payload; read it before trusting or disputing a score.

In [ ]:
import pandas as pd

generation_failures = 0  # generation errors, if any, are counted in Section 4

df = pd.DataFrame(rows)

summary = df.groupby("metric").agg(
    mean_score=("score", "mean"),
    pass_rate=("passed", "mean"),
    cases=("passed", "size"),
).round(3)
print(f"Model under test: {MODEL}")

display(summary)

print("Per-case results:")
with pd.option_context("display.max_colwidth", 120):
    display(df)

## 8. Log to MLflow

First write the local JSON artifact (same as `model_eval_deepeval.ipynb`, kept for git-based reproducibility), then log the run to MLflow via `genai_eval.mlflow_logging.log_eval_run` — the exact function `python -m genai_eval.eval_runner` uses, so notebook runs and CLI runs are consistent in MLflow.

`log_eval_run` logs:
- **Params**: `dataset`, `model_name`, `model_endpoint`, `judge_model`, `judge_prompt_ver`, `metrics`, `n_items`, `n_generation_failures`.
- **Metrics**: `<metric>_mean` and `<metric>_n` for each metric (Correctness, Answer Relevancy).
- **Artifact**: the results JSON file.

`judge_prompt_ver` is derived the same way the CLI runner derives it — DeepEval's metric prompts change with the library version, so the installed version *is* the prompt version (see `CLAUDE.md`'s pinned-judge rule).

In [ ]:
import sys
from datetime import datetime, timezone

sys.path.insert(0, str(REPO_ROOT / "src"))
from genai_eval.mlflow_logging import log_eval_run

# Save the local JSON artifact first (repo convention: reproducibility = Git).
out_dir = REPO_ROOT / "notebooks" / "artifacts" / "deepeval-model-eval"
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / f"results_{MODEL.replace('/', '_')}.json"

with open(out_path, "w", encoding="utf-8") as f:
    json.dump(
        {
            "model": MODEL,
            "base_url": BASE_URL,
            "judge": {"backend": JUDGE_BACKEND, "model": JUDGE_MODEL},
            "temperature": TEMPERATURE,
            "cases": eval_dataset,
            "results": rows,
        },
        f,
        indent=2,
        ensure_ascii=False,
    )
print(f"Saved local artifact -> {out_path}")

# `judge_prompt_ver`: DeepEval's default metric prompts change with the
# library version, so the installed version *is* the prompt version — same
# convention as genai_eval.eval_runner.judge_prompt_version().
import deepeval
judge_prompt_ver = f"deepeval-default@{deepeval.__version__}"

aggregates = {
    metric: {
        "mean": float(summary.loc[metric, "mean_score"]),
        "n": int(summary.loc[metric, "cases"]),
    }
    for metric in summary.index
}

run_payload = {
    "run": {
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "dataset": str(DATASET_PATH),
        "model_endpoint": BASE_URL,
        "model_name": MODEL,
        "judge_endpoint": JUDGE_BASE_URL if JUDGE_BACKEND == "local" else "openai",
        "judge_model": JUDGE_MODEL,
        "judge_prompt_ver": judge_prompt_ver,
        "metrics": ["correctness", "answer_relevancy"],
        "n_items": len(eval_dataset),
        "n_generation_failures": generation_failures,
    },
    "aggregates": aggregates,
}

run_id = log_eval_run(MLFLOW_TRACKING_URI, MLFLOW_EXPERIMENT, run_payload, out_path)
print(f"Logged to MLflow run {run_id} ({MLFLOW_TRACKING_URI}, experiment={MLFLOW_EXPERIMENT!r})")

### Notes

- **Why MLflow, not just a JSON file**: per `CLAUDE.md`, MLflow is the system of record for offline/pre-release eval runs — experiments, golden-dataset runs, and CI regression-gate history all live there so runs are comparable across model/prompt candidates. The local JSON artifact is still written (Section 8) for git-based reproducibility, but it is not the record of truth once logged.
- **Comparing runs in MLflow**: rerun Sections 4–8 with a different `MODEL`/`BASE_URL`, or a different `JUDGE_MODEL`, and each run lands as a separate MLflow run in the same experiment (`MLFLOW_EXPERIMENT`) — compare via the MLflow UI or `mlflow.search_runs()`. **Keep the judge fixed** while comparing models under test; a judge change is logged (`judge_model`, `judge_prompt_ver`) but invalidates any prior baseline per the pinned-judge rule.
- **CLI equivalent**: `python -m genai_eval.eval_runner --dataset datasets/golden_qa_de.jsonl --metrics answer_relevancy,correctness --out results/qa.json` does the same generate-score-log loop non-interactively (e.g. for CI). This notebook is the interactive/exploratory counterpart — same registry-owned metric names, same MLflow logging function.
- **Out of scope here, by design**: no Confident AI cloud login (`deepeval login`) — everything runs and is stored locally / in your own MLflow instance, matching this repo's air-gap-friendly convention.